# 🏌️ Mini Caddie — ONNX to Hailo HEF Compilation
Compile your trained YOLOv8n ONNX model to Hailo HEF format for deployment on Pi 5 + Hailo-8L.

**Uses ClientRunner Python API** (NOT hailomz CLI — which doesn't work on Colab).

## Before you start
1. Download the Hailo Dataflow Compiler (DFC) wheel from https://hailo.ai/developer-zone/
   - Create a free Developer Zone account
   - Select **Hailo-8L** (NOT Hailo-8 or Hailo-10H)
   - Download the Python wheel (Linux x86_64). Pick Python 3.12 if 3.13 isn't offered.
   - Upload the `.whl` file to your Google Drive root
2. Your ONNX model (`mini_caddie_golf_best.onnx`) should already be in Google Drive from training
3. Your dataset zip (`unified-golf-dataset.zip`) should be in Drive (needed for calibration images)
4. Set Colab runtime to **CPU** (Runtime → Change runtime type → None)

## Step 1: Mount Google Drive

In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Hailo DFC + Dependency Fixes
Installs the DFC wheel and fixes three dependency issues:
- **Fix 1a**: `pyparsing<3` (Hailo breaks on pyparsing 3+)
- **Fix 1b**: `httplib2>=0.22` (Colab's default is too old)
- **Fix 1c**: `numpy==1.26.4` (TF expects older numpy ABI)
- **Fix 1d**: `--force-reinstall --no-deps` for DFC wheel (bypasses metadata crash on Python 3.13)

⚠️ After this cell finishes, you MUST **Runtime → Restart session** before continuing.

In [ ]:
# STEP 2: Install DFC + dependency fixes
import os

# Find the DFC wheel in Drive
drive_path = '/content/drive/MyDrive'
whl_files = [f for f in os.listdir(drive_path) if f.endswith('.whl') and 'hailo' in f.lower()]

if whl_files:
    whl_path = os.path.join(drive_path, whl_files[0])
    print(f'Found DFC wheel: {whl_files[0]}')
    # Fix 1d: --force-reinstall --no-deps bypasses metadata-generation-failed on Python 3.13
    !pip install --force-reinstall --no-deps "{whl_path}" -q
    print('DFC installed!')
else:
    print('❌ No Hailo .whl file found in Google Drive!')
    print('Download from https://hailo.ai/developer-zone/ and upload to Drive root.')
    print('Expected: hailo_dataflow_compiler-3.x.x-py3-none-linux_x86_64.whl')

# Fix 1a: Downgrade pyparsing (Hailo DFC breaks on pyparsing 3+)
!pip install 'pyparsing<3' -q

# Fix 1b: Upgrade httplib2 (Colab's default is too old for TF import chain)
!pip install 'httplib2>=0.22' -q

# Fix 1c: Pin numpy to 1.26.4 (TF expects older numpy ABI)
!pip install 'numpy==1.26.4' -q

print()
print('✅ All dependency fixes applied!')
print()
print('⚠️  NOW: Runtime → Restart session')
print('After restart, run Step 3 (NOT Step 2 again — it\'s already installed)')

## Step 3: Post-Restart — Re-mount Drive + Reinstall DFC
⚠️ The runtime restart WIPES all pip installs. You must reinstall the DFC wheel here.

Do NOT restart again after this cell.

In [ ]:
# STEP 3: Post-restart reinstall (runtime restart wipes pip installs!)
import os

# Re-mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Reinstall DFC wheel (--force-reinstall --no-deps)
drive_path = '/content/drive/MyDrive'
whl_files = [f for f in os.listdir(drive_path) if f.endswith('.whl') and 'hailo' in f.lower()]

if whl_files:
    whl_path = os.path.join(drive_path, whl_files[0])
    print(f'Reinstalling DFC: {whl_files[0]}')
    !pip install --force-reinstall --no-deps "{whl_path}" -q
    print('✅ DFC reinstalled!')
else:
    print('❌ No Hailo .whl file found! Check Drive.')

# Re-apply dependency fixes (also wiped by restart)
!pip install 'pyparsing<3' 'httplib2>=0.22' 'numpy==1.26.4' -q
print('✅ Dependency fixes re-applied!')

# Verify the correct module imports
try:
    from hailo_sdk_client import ClientRunner
    print('✅ ClientRunner imported successfully from hailo_sdk_client!')
except ImportError as e:
    print(f'❌ Import failed: {e}')
    print('Check pip list for hailo packages:')
    !pip list 2>/dev/null | grep -i hailo

## Step 4: Unzip Dataset for Calibration Images
Hailo needs sample images to calibrate (quantize) the model during compilation.

In [ ]:
# STEP 4: Unzip dataset for calibration (skips if already extracted)
import zipfile, os

zip_path = '/content/drive/MyDrive/unified-golf-dataset.zip'
extract_path = '/content/dataset'

if os.path.exists(os.path.join(extract_path, 'unified', 'data.yaml')):
    print('✅ Dataset already extracted — skipping!')
else:
    if os.path.exists(zip_path):
        os.makedirs(extract_path, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_path)
        print('✅ Dataset extracted for calibration!')
    else:
        print('❌ Dataset zip not found — upload unified-golf-dataset.zip to Drive')

!ls /content/dataset/unified/ 2>/dev/null || echo 'Dataset not found'

## Step 5: Copy ONNX Model to Working Directory

In [ ]:
# STEP 5: Copy ONNX model
import shutil, os

onnx_src = '/content/drive/MyDrive/mini_caddie_golf_best.onnx'
onnx_dst = '/content/mini_caddie_golf_best.onnx'

if os.path.exists(onnx_src):
    shutil.copy(onnx_src, onnx_dst)
    size_mb = os.path.getsize(onnx_dst) / 1024 / 1024
    print(f'✅ ONNX model copied! Size: {size_mb:.1f} MB')
    # YOLOv8n should be ~12 MB, YOLOv8s is ~42 MB
    if size_mb < 20:
        print('  (Looks like YOLOv8n — good for Hailo-8L!)')
    else:
        print('  ⚠️ Large file — might be YOLOv8s which is too big for Hailo-8L')
else:
    print('❌ ONNX model not found in Drive!')
    print('Make sure mini_caddie_golf_best.onnx is in your Drive root.')

## Step 6: Compile ONNX to HEF (ClientRunner API)
This is the main compilation step. Uses the DFC Python API — NOT the hailomz CLI.

Fixes applied here:
- **Fix 4**: `os.environ['USER'] = 'colab'` (Colab doesn't set USER)
- **Fix 5**: SDKPaths `_is_release = True` monkey-patch (bypasses dist-packages bug)
- **Fix 6**: `end_node_names` cuts off YOLOv8 DFL layer (unsupported by Hailo)
- **Fix 7**: Calibration data as `tf.data.Dataset` with `(image, metadata)` tuples

Takes 5-15 minutes.

In [ ]:
# STEP 6: Compile ONNX to HEF using ClientRunner Python API
import os
os.environ['USER'] = 'colab'  # Fix 4: Colab doesn't set USER env var

import numpy as np
import tensorflow as tf
from PIL import Image
from hailo_sdk_client import ClientRunner  # NOT hailo_sdk_sdk!

onnx_path = '/content/mini_caddie_golf_best.onnx'
hef_output = '/content/mini_caddie_golf.hef'

# --- 6a: Parse ONNX into Hailo internal format ---
# Fix 6: end_node_names cuts off DFL layer (unsupported by Hailo parser)
print('Parsing ONNX model...')
runner = ClientRunner(hw_arch='hailo8l')
runner.translate_onnx_model(
    onnx_path,
    'mini_caddie_golf',  # net_name (2nd positional arg, NOT keyword)
    start_node_names=['images'],
    end_node_names=['/model.22/Sigmoid', '/model.22/dfl/Reshape']
)
print('✅ ONNX parsed successfully!')

# --- 6b: Load calibration images ---
val_dir = '/content/dataset/unified/valid/images'
calib_files = sorted([f for f in os.listdir(val_dir)
                       if f.endswith(('.jpg', '.png', '.jpeg'))])[:100]
print(f'Loading {len(calib_files)} calibration images...')

calib_arrays = []
for f in calib_files:
    img = Image.open(os.path.join(val_dir, f)).resize((640, 640))
    arr = np.array(img, dtype=np.float32)
    calib_arrays.append(arr)

print(f'Loaded {len(calib_arrays)} images, shape: {calib_arrays[0].shape}')

# --- 6c: Optimize (quantize) with calibration data ---
# Fix 7: optimize() requires a callable returning tf.data.Dataset
# with (image, metadata_dict) tuples matching ImageFeed format
print('Optimizing (quantizing) model... this takes a few minutes...')
calib_dataset = tf.data.Dataset.from_tensor_slices(calib_arrays)
calib_dataset = calib_dataset.map(lambda x: (x, {"img_orig": x}))
calib_feed = lambda: calib_dataset

runner.optimize(calib_feed)
print('✅ Model optimized!')

# --- 6d: Patch SDKPaths before compile ---
# Fix 5: SDKPaths._is_release must be True on Colab (dist-packages vs site-packages)
print('Patching SDKPaths...')
from hailo_sdk_common.paths_manager.paths import SDKPaths
paths = SDKPaths()
paths._is_release = True
print('✅ SDKPaths patched!')

# --- 6e: Compile to HEF ---
print('Compiling to HEF... this takes 5-15 minutes...')
runner.load_model_script("""
normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])
performance_param(compiler_optimization_level=max, optimize_for_power=True)
""")

hef = runner.compile()

with open(hef_output, 'wb') as f:
    f.write(hef)

if os.path.exists(hef_output):
    size_mb = os.path.getsize(hef_output) / 1024 / 1024
    print(f'✅ HEF compiled! Size: {size_mb:.1f} MB')
    print(f'Saved to: {hef_output}')
else:
    print('❌ HEF file not found — compilation may have failed')

## Step 7: Save HEF to Google Drive

In [ ]:
# STEP 7: Save HEF to Google Drive
import shutil, os

hef_src = '/content/mini_caddie_golf.hef'
hef_dst = '/content/drive/MyDrive/mini_caddie_golf.hef'

if os.path.exists(hef_src):
    shutil.copy(hef_src, hef_dst)
    print('✅ HEF saved to Google Drive!')
else:
    print('❌ HEF file not found — did compilation succeed?')

## Step 8: Create labels.json for Deployment
Hailo adds a background class at index 0 — all class IDs shift by +1.

In [ ]:
# STEP 8: Create labels.json
import json

labels = {
    "0": "background",
    "1": "golf_ball",
    "2": "golf_club",
    "3": "golf_club_head",
    "4": "golf_hole",
    "5": "golf_mat",
    "6": "person",
    "7": "player_not_ready",
    "8": "player_ready"
}

labels_path = '/content/drive/MyDrive/mini_caddie_labels.json'
with open(labels_path, 'w') as f:
    json.dump(labels, f, indent=2)

print('✅ labels.json saved to Drive!')
print(json.dumps(labels, indent=2))

## ✅ After compilation completes
Your Google Drive should now have:
- `mini_caddie_golf.hef` — the compiled Hailo model (deploy this to Pi)
- `mini_caddie_labels.json` — class labels (with background at 0)
- `mini_caddie_golf_best.onnx` — original ONNX (backup)
- `mini_caddie_golf_best.pt` — PyTorch weights (backup)

## Next: Deploy on Pi
1. Download `mini_caddie_golf.hef` and `mini_caddie_labels.json` from Drive
2. Push to GitHub (Mac) → clone on Pi (file shuttle for restricted networks)
3. Run: `hailo-detect --hef-path mini_caddie_golf.hef --labels-json mini_caddie_labels.json --input rpi`